# Centralised — Draft-Revision Objective

$$\mathcal{L} = \mathcal{L}_{\mathrm{gen}} + \lambda_{\mathrm{edit}} \cdot \mathcal{L}_{\mathrm{rev}}$$

Two supervised tasks sharing the target $y^{+}$:

- **generation** $(q, c^{+}) \rightarrow y^{+}$ — identical to SFT
- **revision** $(q, c^{+}, y^{-}) \rightarrow y^{+}$ — the general answer is supplied as a draft to personalise

`TRAIN_MODE = "weighted"`, `LAMBDA_EDIT = 0.3`. At inference the model is
prompted with the generation template only, in a single pass.

---

*Notation:* `q` query · `c+` relevant snippet · `c-` off-topic snippet from the same user · `y+` personalised answer · `y-` general answer


In [1]:
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive")
    !pip -q install -U datasets huggingface_hub transformers accelerate peft bitsandbytes rouge-score
else:
    DRIVE_ROOT = Path(".")

if IN_COLAB:
    PP_ROOT = DRIVE_ROOT / "privacy_perserving_pllm"
else:
    _here = Path(".").resolve()
    PP_ROOT = next(
        (
            p for p in [_here, *_here.parents]
            if (p / "v2_personamem_persona_subsets").exists()
            or all((p / d).exists() for d in ("centralised", "evaluation", "fed_grad_avg", "zero_shot"))
        ),
        _here,
    )
SUBSET_NAME = "all_subset"
SUBSETS_DIR = PP_ROOT / "v2_personamem_persona_subsets"

RESULTS_DIR = PP_ROOT / SUBSET_NAME / "v2_personamem_centralized_gen_edit_snippet"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print("Subset:", SUBSET_NAME)
print("Results will be saved to:", RESULTS_DIR.resolve())

Mounted at /content/drive
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 35.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 771.9/771.9 kB 72.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 241.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 94.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 69.9 MB/s eta 0:00:00
Subset: all_subset
Results will be saved to: /content/drive/MyDrive/privacy_perserving_pllm/all_subset/v2_personamem_centralized_gen_edit_snippet


In [2]:
import ast
import json
import random
from typing import Any, Dict, List

import pandas as pd
from datasets import load_dataset
from transformers import AutoTokenizer

DATASET_NAME = "bowen-upenn/PersonaMem-v2"
CONFIG = "benchmark"
TEXT_SPLITS = ("train_text", "val_text", "benchmark_text")

SEED = 42
TOKENIZER_NAME = "Qwen/Qwen3-0.6B"
MODEL_NAME = "Qwen/Qwen3-0.6B"
MAX_SEQ_LEN = 4096
MAX_SNIPPET_TOKENS = 2048
MAX_ANSWER_TOKENS = 512
MAX_NEW_TOKENS = 512

TRAIN_MODE = "weighted"
LAMBDA_EDIT = 0.3
EDIT_MIX_RATIO = 0.3

START_MODE = "hf"
CONTINUE_FROM_EPOCH = 3
CONTINUE_EPOCHS = 1
GLOBAL_EPOCHS = 3
LOCAL_LR = 2e-4
LOCAL_BATCH_SIZE = 8
EVAL_BATCH_SIZE = 32
EVAL_TRAIN = False

random.seed(SEED)

def load_split(split):
    if split not in TEXT_SPLITS:
        raise ValueError(f"split must be one of {TEXT_SPLITS}, got {split!r}")
    return load_dataset(DATASET_NAME, CONFIG, split=split)

def parse_user_query(raw):
    if isinstance(raw, dict):
        return str(raw.get("content", raw)).strip()
    if isinstance(raw, str):
        try:
            d = ast.literal_eval(raw)
            if isinstance(d, dict):
                return str(d.get("content", raw)).strip()
        except (ValueError, SyntaxError):
            pass
        return raw.strip()
    return str(raw).strip()

def parse_incorrect_answers(raw):
    if isinstance(raw, list):
        return [str(x) for x in raw]
    if hasattr(raw, "tolist"):
        return [str(x) for x in raw.tolist()]
    if isinstance(raw, str):
        try:
            val = ast.literal_eval(raw)
            if isinstance(val, list):
                return [str(x) for x in val]
        except (ValueError, SyntaxError):
            pass
        return [raw]
    return []

def get_snippet(row):
    val = row.get("related_conversation_snippet")
    if val is None:
        return ""
    return str(val).strip()

In [3]:
def load_subset(name, subsets_dir=SUBSETS_DIR):
    """Load a persona subset (ids + train/val rows) saved by create_persona_subsets.ipynb."""
    subset_dir = Path(subsets_dir) / name
    if not subset_dir.exists():
        raise FileNotFoundError(f"Subset not found: {subset_dir}. Run create_persona_subsets.ipynb first.")
    with open(subset_dir / "personas.json", "r", encoding="utf-8") as f:
        persona_ids = [int(p) for p in json.load(f)]
    tr = pd.read_parquet(subset_dir / "train.parquet")
    va = pd.read_parquet(subset_dir / "val.parquet")
    return sorted(persona_ids), tr, va

MIN_VAL_ROWS = 4
CLIENT_PERSONAS, train_df, val_df = load_subset(SUBSET_NAME)
NUM_CLIENTS = len(CLIENT_PERSONAS)
val_counts = val_df.groupby("persona_id").size()
print(f"train rows: {len(train_df):,} | val rows: {len(val_df):,}")
print(f"Subset '{SUBSET_NAME}': {NUM_CLIENTS} personas")
print(f"Personas ({len(CLIENT_PERSONAS)}): first 10 = {CLIENT_PERSONAS[:10]} ...")
print(f"Val rows per selected persona (min/mean/max): "
      f"{val_counts[CLIENT_PERSONAS].min()}/{val_counts[CLIENT_PERSONAS].mean():.1f}/{val_counts[CLIENT_PERSONAS].max()}")

train rows: 3,870 | val rows: 714
Subset 'all_subset': 150 personas
Personas (150): first 10 = [6, 9, 14, 18, 33, 41, 51, 56, 57, 73] ...
Val rows per selected persona (min/mean/max): 4/4.8/8


In [4]:
import os

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import gc

import torch
from datasets import Dataset, concatenate_datasets
from peft import (
    LoraConfig,
    PeftModel,
    get_peft_model,
    get_peft_model_state_dict,
    prepare_model_for_kbit_training,
    set_peft_model_state_dict,
)
from transformers import AutoModelForCausalLM, BitsAndBytesConfig, DataCollatorForSeq2Seq


model_tok = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if model_tok.pad_token is None:
    model_tok.pad_token = model_tok.eos_token
model_tok.padding_side = "right"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

collator = DataCollatorForSeq2Seq(model_tok, label_pad_token_id=-100, padding=True, return_tensors="pt")
KEEP_COLS = ["user_query", "correct_answer", "incorrect_answers", "related_conversation_snippet"]
EDIT_INSTRUCTION = (
    "Revise the draft so all personal details are supported by the context. "
    "Correct any unsupported personal facts while keeping a helpful answer."
)


def build_system_prompt():
    return (
        "You are a personalised assistant. Use the conversation snippet to find "
        "information or connections relevant to the question, then provide the answer."
    )


def truncate_snippet(snippet, max_tokens):
    ids = model_tok(snippet, add_special_tokens=False)["input_ids"]
    if len(ids) <= max_tokens:
        return snippet
    return model_tok.decode(ids[-max_tokens:], skip_special_tokens=True)


def truncate_text(text, max_tokens):
    ids = model_tok(text, add_special_tokens=False)["input_ids"]
    if len(ids) <= max_tokens:
        return text
    return model_tok.decode(ids[:max_tokens], skip_special_tokens=True)


def build_generation_user_content(row):
    """Same format as centralized_sft_snippet (single-pass inference compatible)."""
    snippet = truncate_snippet(get_snippet(row), MAX_SNIPPET_TOKENS)
    return (
        "Relevant snippet from our earlier conversation:\n"
        f"{snippet}\n\n"
        f"{parse_user_query(row['user_query'])}"
    )


def build_edit_user_content(row, draft):
    snippet = truncate_snippet(get_snippet(row), MAX_SNIPPET_TOKENS)
    draft = truncate_text(str(draft).strip(), MAX_ANSWER_TOKENS)
    return (
        "Relevant snippet from our earlier conversation:\n"
        f"{snippet}\n\n"
        f"Question:\n{parse_user_query(row['user_query'])}\n\n"
        f"Draft answer:\n{draft}\n\n"
        f"Instruction:\n{EDIT_INSTRUCTION}"
    )


def tokenize_prompt_answer(prompt_text, answer_text):
    prompt_ids = model_tok(prompt_text, add_special_tokens=False)["input_ids"]
    answer_ids = model_tok(
        str(answer_text) + model_tok.eos_token,
        add_special_tokens=False, truncation=True, max_length=MAX_ANSWER_TOKENS,
    )["input_ids"]
    input_ids = (prompt_ids + answer_ids)[:MAX_SEQ_LEN]
    labels = ([-100] * len(prompt_ids) + answer_ids)[:MAX_SEQ_LEN]
    return {
        "input_ids": input_ids,
        "labels": labels,
        "attention_mask": [1] * len(input_ids),
    }


def make_generation_tokenize_fn(system_prompt):
    def fn(example):
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": build_generation_user_content(example)},
        ]
        prompt_text = model_tok.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        return tokenize_prompt_answer(prompt_text, example["correct_answer"])
    return fn


def make_edit_tokenize_fn(system_prompt):
    def fn(example):
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": build_edit_user_content(example, example["draft_answer"])},
        ]
        prompt_text = model_tok.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        return tokenize_prompt_answer(prompt_text, example["correct_answer"])
    return fn


def rows_with_draft(rows):
    out = []
    for row in rows:
        incorrects = parse_incorrect_answers(row.get("incorrect_answers"))
        if not incorrects:
            continue
        item = dict(row)
        item["draft_answer"] = incorrects[0]
        out.append(item)
    return out


system_prompt = build_system_prompt()
client_data: Dict[Any, Dict[str, Any]] = {}
gen_parts = []
edit_parts = []

for pid in CLIENT_PERSONAS:
    p_train = train_df[train_df["persona_id"] == pid].reset_index(drop=True)
    rows = p_train[KEEP_COLS].to_dict("records")
    edit_rows = rows_with_draft(rows)

    gen_ds = Dataset.from_pandas(p_train[KEEP_COLS].reset_index(drop=True))
    gen_tok = gen_ds.map(make_generation_tokenize_fn(system_prompt), remove_columns=gen_ds.column_names)
    gen_parts.append(gen_tok)

    if edit_rows:
        edit_ds = Dataset.from_list(edit_rows)
        edit_tok = edit_ds.map(make_edit_tokenize_fn(system_prompt), remove_columns=edit_ds.column_names)
        edit_parts.append(edit_tok)
    else:
        edit_tok = Dataset.from_list([])

    client_data[pid] = {
        "n_gen": len(gen_tok),
        "n_edit": len(edit_tok),
        "system_prompt": system_prompt,
    }
    print(f"  persona {pid}: {len(gen_tok)} generation | {len(edit_tok)} editing")

pooled_gen = concatenate_datasets(gen_parts)
pooled_edit = concatenate_datasets(edit_parts) if edit_parts else Dataset.from_list([])
print(f"Pooled generation examples: {len(pooled_gen)}")
print(f"Pooled editing examples:    {len(pooled_edit)}")
print(f"TRAIN_MODE={TRAIN_MODE} | LAMBDA_EDIT={LAMBDA_EDIT} | EDIT_MIX_RATIO={EDIT_MIX_RATIO}")

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

...

Map:   0%|          | 0/21 [00:00<?, ? examples/s]

  persona 987: 21 generation | 21 editing


Map:   0%|          | 0/23 [00:00<?, ? examples/s]

Map:   0%|          | 0/23 [00:00<?, ? examples/s]

  persona 996: 23 generation | 23 editing
Pooled generation examples: 3870
Pooled editing examples:    3869
TRAIN_MODE=weighted | LAMBDA_EDIT=0.3 | EDIT_MIX_RATIO=0.3


In [5]:
from itertools import cycle
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

def load_continual_adapter(from_epoch):
    """Continual learning: load base model + a previously saved LoRA adapter (trainable)."""
    adapter_dir = RESULTS_DIR / "global" / f"adapter_epoch_{from_epoch}"
    if not adapter_dir.exists():
        raise FileNotFoundError(f"Adapter to continue from not found: {adapter_dir}")
    base = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, quantization_config=bnb_config, device_map="auto",
        trust_remote_code=True, attn_implementation="sdpa",
    )
    base.config.use_cache = False
    base = prepare_model_for_kbit_training(base, use_gradient_checkpointing=True)
    base.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    base.enable_input_require_grads()
    m = PeftModel.from_pretrained(base, str(adapter_dir), is_trainable=True)
    m.print_trainable_parameters()
    return m

def load_base_with_adapter():
    base = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, quantization_config=bnb_config, device_map="auto",
        trust_remote_code=True, attn_implementation="sdpa",
    )
    base.config.use_cache = False
    base = prepare_model_for_kbit_training(base, use_gradient_checkpointing=True)
    base.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    base.enable_input_require_grads()
    m = get_peft_model(base, lora_config)
    return m

def clone_state(model):
    return {k: v.detach().cpu().clone() for k, v in get_peft_model_state_dict(model).items()}

def build_mixed_dataset(gen_ds, edit_ds, edit_ratio=EDIT_MIX_RATIO, seed=SEED):
    """Option A: mix generation + editing examples. edit_ratio is the edit fraction."""
    if len(edit_ds) == 0:
        print("warning: no edit examples; falling back to generation-only mix")
        return gen_ds
    edit_ratio = float(edit_ratio)
    if not (0.0 < edit_ratio < 1.0):
        raise ValueError(f"EDIT_MIX_RATIO must be in (0,1), got {edit_ratio}")
    n_gen = len(gen_ds)
    n_edit_target = max(1, int(round(n_gen * edit_ratio / (1.0 - edit_ratio))))
    rng = random.Random(seed)
    edit_indices = list(range(len(edit_ds)))
    if n_edit_target <= len(edit_ds):
        chosen = rng.sample(edit_indices, n_edit_target)
    else:
        chosen = [rng.choice(edit_indices) for _ in range(n_edit_target)]
    edit_sample = edit_ds.select(chosen)
    mixed = concatenate_datasets([gen_ds, edit_sample]).shuffle(seed=seed)
    print(
        f"Mixed dataset: {len(gen_ds)} gen + {len(edit_sample)} edit "
        f"= {len(mixed)} total "
        f"(edit fraction={len(edit_sample)/len(mixed):.2f})"
    )
    return mixed

def sft_train(model, tokenized, epochs, lr, batch_size=LOCAL_BATCH_SIZE):
    """Ordinary SFT over a single (possibly mixed) dataset — Option A."""
    model.train()
    model.config.use_cache = False
    loader = DataLoader(tokenized, batch_size=batch_size, shuffle=True, collate_fn=collator)
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(params, lr=lr)
    epoch_losses: List[float] = []
    n_batches = 0
    for _ in range(epochs):
        batch_losses: List[float] = []
        for batch in tqdm(loader, desc="Training Batch"):
            batch = {k: v.to(model.device) for k, v in batch.items()}
            with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
                loss = model(**batch).loss
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            batch_losses.append(float(loss.detach().cpu()))
            n_batches += 1
        if batch_losses:
            epoch_losses.append(sum(batch_losses) / len(batch_losses))
    del optimizer
    gc.collect()
    torch.cuda.empty_cache()
    mean_loss = sum(epoch_losses) / len(epoch_losses) if epoch_losses else None
    return {
        "mean_loss": mean_loss,
        "mean_gen_loss": mean_loss,
        "mean_edit_loss": None,
        "epoch_losses": epoch_losses,
        "n_batches": n_batches,
    }

def weighted_gen_edit_train(model, gen_tokenized, edit_tokenized, epochs, lr, lambda_edit=LAMBDA_EDIT, batch_size=LOCAL_BATCH_SIZE):
    """Option B: each step = generation_loss + lambda_edit * edit_loss."""
    if len(edit_tokenized) == 0:
        print("warning: no edit examples; falling back to generation-only SFT")
        return sft_train(model, gen_tokenized, epochs, lr, batch_size)

    model.train()
    model.config.use_cache = False
    gen_loader = DataLoader(gen_tokenized, batch_size=batch_size, shuffle=True, collate_fn=collator)
    edit_loader = DataLoader(edit_tokenized, batch_size=batch_size, shuffle=True, collate_fn=collator)
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(params, lr=lr)

    epoch_losses: List[float] = []
    epoch_gen_losses: List[float] = []
    epoch_edit_losses: List[float] = []
    n_batches = 0

    for _ in range(epochs):
        edit_iter = cycle(edit_loader)
        batch_losses: List[float] = []
        gen_losses: List[float] = []
        edit_losses: List[float] = []
        for gen_batch in tqdm(gen_loader, desc="Training Batch"):
            edit_batch = next(edit_iter)
            gen_batch = {k: v.to(model.device) for k, v in gen_batch.items()}
            edit_batch = {k: v.to(model.device) for k, v in edit_batch.items()}
            optimizer.zero_grad()
            with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
                generation_loss = model(**gen_batch).loss
                edit_loss = model(**edit_batch).loss
                total_loss = generation_loss + lambda_edit * edit_loss
            total_loss.backward()
            optimizer.step()
            batch_losses.append(float(total_loss.detach().cpu()))
            gen_losses.append(float(generation_loss.detach().cpu()))
            edit_losses.append(float(edit_loss.detach().cpu()))
            n_batches += 1
        if batch_losses:
            epoch_losses.append(sum(batch_losses) / len(batch_losses))
            epoch_gen_losses.append(sum(gen_losses) / len(gen_losses))
            epoch_edit_losses.append(sum(edit_losses) / len(edit_losses))

    del optimizer
    gc.collect()
    torch.cuda.empty_cache()
    return {
        "mean_loss": sum(epoch_losses) / len(epoch_losses) if epoch_losses else None,
        "mean_gen_loss": sum(epoch_gen_losses) / len(epoch_gen_losses) if epoch_gen_losses else None,
        "mean_edit_loss": sum(epoch_edit_losses) / len(epoch_edit_losses) if epoch_edit_losses else None,
        "epoch_losses": epoch_losses,
        "epoch_gen_losses": epoch_gen_losses,
        "epoch_edit_losses": epoch_edit_losses,
        "n_batches": n_batches,
    }

def train_one_epoch(model):
    if TRAIN_MODE == "mix":
        mixed = build_mixed_dataset(pooled_gen, pooled_edit, EDIT_MIX_RATIO, SEED)
        return sft_train(model, mixed, 1, LOCAL_LR)
    if TRAIN_MODE == "weighted":
        return weighted_gen_edit_train(
            model, pooled_gen, pooled_edit, 1, LOCAL_LR, LAMBDA_EDIT, LOCAL_BATCH_SIZE
        )
    raise ValueError(f"TRAIN_MODE must be 'weighted' or 'mix', got {TRAIN_MODE!r}")

In [6]:
if START_MODE == "adapter":
    model = load_continual_adapter(CONTINUE_FROM_EPOCH)
    _start_epoch = CONTINUE_FROM_EPOCH + 1
    _end_epoch = CONTINUE_FROM_EPOCH + CONTINUE_EPOCHS
    print(f"Continual learning from adapter_epoch_{CONTINUE_FROM_EPOCH}: "
          f"training epochs {_start_epoch}..{_end_epoch}")
else:
    model = load_base_with_adapter()
    _start_epoch = 1
    _end_epoch = GLOBAL_EPOCHS
    print(f"Fresh HuggingFace base model: training epochs {_start_epoch}..{_end_epoch}")

GLOBAL_DIR = RESULTS_DIR / "global"
GLOBAL_DIR.mkdir(parents=True, exist_ok=True)
epoch_adapter_dirs: List[str] = []
centralized_loss_log: Dict[str, Any] = {
    "method": "centralized_gen_edit_global",
    "train_mode": TRAIN_MODE,
    "lambda_edit": LAMBDA_EDIT,
    "edit_mix_ratio": EDIT_MIX_RATIO,
    "epochs": [],
}

for epoch in range(_start_epoch, _end_epoch + 1):
    epoch_loss = train_one_epoch(model)
    epoch_entry = {
        "epoch": epoch,
        "pooled_mean_loss": epoch_loss["mean_loss"],
        "pooled_mean_gen_loss": epoch_loss.get("mean_gen_loss"),
        "pooled_mean_edit_loss": epoch_loss.get("mean_edit_loss"),
        "pooled_epoch_losses": epoch_loss["epoch_losses"],
        "pooled_n_batches": epoch_loss["n_batches"],
        "train_mode": TRAIN_MODE,
    }
    centralized_loss_log["epochs"].append(epoch_entry)
    with open(GLOBAL_DIR / f"losses_epoch_{epoch}.json", "w", encoding="utf-8") as f:
        json.dump(epoch_entry, f, indent=2)

    epoch_dir = GLOBAL_DIR / f"adapter_epoch_{epoch}"
    model.save_pretrained(str(epoch_dir))
    model_tok.save_pretrained(str(epoch_dir))
    epoch_adapter_dirs.append(str(epoch_dir))
    loss_str = f"{epoch_entry['pooled_mean_loss']:.4f}" if epoch_entry["pooled_mean_loss"] is not None else "n/a"
    gen_str = (
        f"{epoch_entry['pooled_mean_gen_loss']:.4f}"
        if epoch_entry["pooled_mean_gen_loss"] is not None else "n/a"
    )
    edit_str = (
        f"{epoch_entry['pooled_mean_edit_loss']:.4f}"
        if epoch_entry["pooled_mean_edit_loss"] is not None else "n/a"
    )
    print(
        f"Epoch {epoch}/{_end_epoch} done | total={loss_str} | "
        f"gen={gen_str} | edit={edit_str} | saved {epoch_dir.name}"
    )

with open(GLOBAL_DIR / "training_losses.json", "w", encoding="utf-8") as f:
    json.dump(centralized_loss_log, f, indent=2)

global_state = clone_state(model)

global_adapter_dir = GLOBAL_DIR / "adapter"
model.save_pretrained(str(global_adapter_dir))
model_tok.save_pretrained(str(global_adapter_dir))

with open(GLOBAL_DIR / "config.json", "w", encoding="utf-8") as f:
    json.dump({
        "method": "centralized_gen_edit_global",
        "subset": SUBSET_NAME,
        "client_personas": [int(p) for p in CLIENT_PERSONAS],
        "num_clients": len(CLIENT_PERSONAS),
        "min_val_rows": MIN_VAL_ROWS,
        "seed": SEED,
        "global_epochs": GLOBAL_EPOCHS,
        "start_mode": START_MODE,
        "continue_from_epoch": CONTINUE_FROM_EPOCH if START_MODE == "adapter" else None,
        "continue_epochs": CONTINUE_EPOCHS if START_MODE == "adapter" else None,
        "trained_epoch_range": [int(_start_epoch), int(_end_epoch)],
        "train_mode": TRAIN_MODE,
        "lambda_edit": LAMBDA_EDIT,
        "edit_mix_ratio": EDIT_MIX_RATIO,
        "pooled_generation_examples": len(pooled_gen),
        "pooled_editing_examples": len(pooled_edit),
        "local_lr": LOCAL_LR,
        "local_batch_size": LOCAL_BATCH_SIZE,
        "epoch_adapter_dirs": epoch_adapter_dirs,
        "final_adapter_dir": str(global_adapter_dir),
    }, f, indent=2)

print("Saved final global centralized gen+edit adapter to:", global_adapter_dir.resolve())

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.50GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Fresh HuggingFace base model: training epochs 1..3


...

Epoch 1/3 done | total=2.8741 | gen=2.3991 | edit=1.5835 | saved adapter_epoch_1


Training Batch:   0%|          | 0/484 [00:00<?, ?it/s]

Epoch 2/3 done | total=2.2523 | gen=1.8951 | edit=1.1905 | saved adapter_epoch_2


Training Batch:   0%|          | 0/484 [00:00<?, ?it/s]

Epoch 3/3 done | total=1.6432 | gen=1.3926 | edit=0.8354 | saved adapter_epoch_3
Saved final global centralized gen+edit adapter to: /content/drive/MyDrive/privacy_perserving_pllm/all_subset/v2_personamem_centralized_gen_edit_snippet/global/adapter


In [7]:
gc.collect()
torch.cuda.empty_cache()

In [ ]:
@torch.no_grad()
def generate_answer(model, tok, user_content, system_prompt=None, max_new_tokens=MAX_NEW_TOKENS):
    system_prompt = system_prompt or build_system_prompt()
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_content},
    ]
    prompt_text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok(prompt_text, return_tensors="pt", add_special_tokens=False).to(model.device)
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        pad_token_id=tok.eos_token_id,
    )
    new_tokens = out[0, inputs["input_ids"].shape[1]:]
    return tok.decode(new_tokens, skip_special_tokens=True).strip()

@torch.no_grad()
def two_pass_generate(model, tok, row, system_prompt=None):
    """Generate then edit the model's own draft for better context consistency."""
    system_prompt = system_prompt or build_system_prompt()
    y1 = generate_answer(model, tok, build_generation_user_content(row), system_prompt)
    draft_row = dict(row)
    y2 = generate_answer(model, tok, build_edit_user_content(draft_row, y1), system_prompt)
    return {"draft": y1, "final": y2}

def load_global_model(epoch=None):
    """Load the centralized global LoRA adapter. Use epoch=1..GLOBAL_EPOCHS for checkpoints."""
    if epoch is not None:
        adapter_dir = RESULTS_DIR / "global" / f"adapter_epoch_{epoch}"
    else:
        adapter_dir = RESULTS_DIR / "global" / "adapter"
    base = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, quantization_config=bnb_config, device_map="auto",
        trust_remote_code=True, attn_implementation="sdpa",
    )
    base.config.use_cache = True
    m = PeftModel.from_pretrained(base, str(adapter_dir))
    m.eval()
    tok = AutoTokenizer.from_pretrained(str(adapter_dir))
    return m, tok

print("Final global adapter:", RESULTS_DIR / "global" / "adapter")
print("Per-epoch checkpoints:", RESULTS_DIR / "global" / "adapter_epoch_<n>")
print("Method key for shared inference:", "centralized_gen_edit_snippet")